# Behavioral AML GNN (PayPal Topology)

This notebook demonstrates the **Behavioral GNN** approach designed for environments where counterparty information is missing (e.g., all transactions are to/from PayPal).

## 1. Introduction: Nodes & Features

Instead of linking Customers to specific beneficiaries, we link them to **Behavioral Nodes**. This transforms "similarity of action" into "graph connectivity".

### Node Types
- **`Customer`**: The main actors. Features can be user-defined (e.g., total volume, ratio).
- **`Bank`**: The funding source institution. Useful for catching rings targeting specific bank KYC weaknesses.
- **`Date`**: Absolute calendar dates. Captures synchronized mass-activation events.
- **`DayOfMonth`**: Days 1-31. Captures cyclical patterns like payroll or benefit fraud.
- **`DayOfWeek`**: Monday-Sunday. Captures operational schedules (e.g., "Friday Afternoon Structuring").
- **`AmountBin`**: Discretized transaction amounts. Specifically tuned to catch structuring (e.g., $9,900 range).

### Edge Weights
- Connections are weighted by **Log-Transformed Transaction Amount**. This ensures that heavy money flows carry more "signal" through the network than small transactions.

--- 

## 2. Parameter Tuning Guide

### Graph Construction Parameters
- **`customer_feature_cols`**: (List of strings). Define which columns in your data should be treated as customer node features. The model will automatically normalize these.
- **`date_window`**: (Integer). Controls "fuzzy" temporal matching (T +/- N days).
- **`date_granularity`**: ('day', 'week', 'month'). Aggregates time nodes to coarser buckets.
- **`amount_bins`**: (List of floats). Defines the resolution of amount matching.

### Model & Detection Parameters
- **`architecture`**: ('sage' or 'gat'). 'sage' (using GraphConv) is robust and uses edge weights. 'gat' uses attention weights.
- **`hidden_channels`**: (Integer, e.g., 32, 64). Complexity of the learned embeddings.
- **`epochs`**: (Integer). Training duration. Watch the loss curve.
- **`contamination`**: (Float, 0.0 to 0.5). The expected percentage of anomalies in the data.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.gnn.modeling.behavior_gnn import BehaviorGraphBuilder, BehavioralGNN, train_behavior_gnn, detect_behavioral_anomalies, set_seed, find_strongly_connected_peers
from scripts.visualize_behavioral_graph import visualize_suspect

# Global seed init
set_seed(42)

print("Modules loaded.")

## 3. Generate Mock Data with Features

We generate data including a custom feature `risk_score_external`.

In [ ]:
np.random.seed(42)
n_total = 500

df = pd.DataFrame({
    'cust_id': [f'User_{i}' for i in range(n_total)],
    'amount': np.random.exponential(1000, n_total),
    'date': pd.date_range('2024-01-01', periods=n_total, freq='h'),
    'bank': np.random.choice(['Chase', 'Wells', 'BoA'], n_total),
    'risk_score_external': np.random.rand(n_total), 
    'tenure_months': np.random.randint(1, 120, n_total) 
})

# Inject a synthetic Mule cluster
df.loc[0:10, 'amount'] = 9500
df.loc[0:10, 'date'] = '2024-01-05'
df.loc[0:10, 'bank'] = 'SketchyBank'

print(f"Data Generated: {len(df)} txns.")

## 4. Build Weighted Graph with Custom Features

In [ ]:
config = {
    'customer': 'cust_id',
    'amount': 'amount',
    'date': 'date',
    'bank': 'bank',
    
    # --- New Flexible Features ---
    'customer_feature_cols': ['risk_score_external', 'tenure_months'],
    'date_window': 1,
    'amount_bins': [-1, 1000, 5000, 9000, 10000, float('inf')]
}

set_seed(42) # Ensure random node features (embeddings) are consistent per run
builder = BehaviorGraphBuilder(config)
data = builder.build(df)

print("Graph Structure (Note edge_weight existence):")
print(data)

## 5. Training & Anomaly Detection

In [ ]:
set_seed(42) # Ensure weight initialization and negative sampling are consistent
model = BehavioralGNN(data.metadata(), architecture='sage') 

print("Training Weighted GNN...")
model = train_behavior_gnn(model, data, epochs=30)

results = detect_behavioral_anomalies(model, data, builder.cust_map, contamination=0.05)
print("Done.")
results.sort_values('risk_score', ascending=False).head(10)

## 6. Visualization

In [ ]:
top_suspect = results.sort_values('risk_score', ascending=False).iloc[0]['customer_id']
visualize_suspect(top_suspect, data, builder, hops=2, risk_df=results, max_associates=5)

## 7. Syndicate Identification

We can explicitly list the customers who share multiple behavioral patterns, indicating they are part of the same ring.

In [ ]:
print("Finding connected peers (Syndicates)...")
# Look for pairs sharing at least 3 behaviors (e.g. Date + Amount + Bank)
syndicates = find_strongly_connected_peers(data, builder.cust_map, min_shared_nodes=3)

print("Top Connected Pairs:")
print(syndicates.head(10))